# 002 — Upload Data (Postgres OLTP + S3)

Load the parquet outputs of `001-prepare-dataset.ipynb` (MovieLens), merge movie
metadata into the interactions, apply a **temporal holdout**, then:

1. Upload OLTP rows → **Postgres** (`movie_ratings` table, local Docker or AWS RDS).
2. Upload the holdout parquet (interactions + metadata) → **S3 / MinIO**.

Mirrors the reference `02-upload-data.ipynb` (Amazon Reviews). For MovieLens we keep
the epoch timestamp column and derive a naive datetime for the OLTP `TIMESTAMP`.

## Setup

In [1]:
import os
import sys
import json

import numpy as np
from dotenv import load_dotenv

load_dotenv(os.path.join("..", ".env"))

import boto3
import pandas as pd
from loguru import logger
from pydantic import BaseModel
from sqlalchemy import create_engine, text

sys.path.insert(0, "..")
from src.data_prep_utils import basic_report

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

In [2]:
class Args(BaseModel):
    run_name: str = "002-upload-data"
    random_seed: int = int(os.getenv("RANDOM_SEED", "42"))

    # Inputs (written by 001-prepare-dataset)
    train_fp: str = ""
    val_fp: str = ""
    meta_fp: str = ""
    holdout_fp: str = "holdout.parquet"  # local temp file before S3 upload

    # MovieLens interaction schema
    user_col: str = "userId"
    item_col: str = "movieId"
    rating_col: str = "rating"
    timestamp_col: str = "timestamp"  # epoch seconds in the parquet

    # Number of days held out of the OLTP so they can later be simulated as new data
    num_days_holdout: int = 30

    # Postgres / RDS
    use_local: bool = os.getenv("DATA_UPLOAD_USE_LOCAL", "true").lower() == "true"
    pg_user: str = os.getenv("PG_USER", "")
    pg_password: str = os.getenv("PG_PASSWORD", "")
    pg_host: str = os.getenv("PG_HOST", "localhost")
    pg_port: str = os.getenv("PG_PORT", "5435")
    pg_db: str = os.getenv("PG_DB", "postgres")
    pg_schema: str = os.getenv("PG_SCHEMA", "public")
    table_name: str = os.getenv("PG_TABLE", "movie_ratings")

    # S3 / MinIO
    aws_access_key_id: str = os.getenv("AWS_ACCESS_KEY_ID", "")
    aws_secret_access_key: str = os.getenv("AWS_SECRET_ACCESS_KEY", "")
    aws_region: str = os.getenv("AWS_DEFAULT_REGION", "ap-southeast-1")
    s3_bucket: str = os.getenv("S3_BUCKET", "recsys-ops")
    s3_endpoint: str = os.getenv("S3_ENDPOINT", "http://localhost:9000")

    def init(self):
        persist_dir = os.getenv("NOTEBOOK_PERSIST_DIR", "data/001-prepare-dataset")
        base = os.path.abspath(persist_dir)
        self.train_fp = os.path.join(base, "train.parquet")
        self.val_fp = os.path.join(base, "val.parquet")
        self.meta_fp = os.path.join(base, "raw_meta.parquet")
        return self


def _mask(s: str) -> str:
    if not s:
        return "(empty)"
    return s[:2] + "***" if len(s) > 2 else "***"


args = Args().init()
safe = {
    "use_local": args.use_local,
    "train_fp": args.train_fp,
    "val_fp": args.val_fp,
    "meta_fp": args.meta_fp,
    "num_days_holdout": args.num_days_holdout,
    "table_name": args.table_name,
    "pg_host": args.pg_host,
    "pg_port": args.pg_port,
    "pg_db": args.pg_db,
    "pg_user": args.pg_user,
    "pg_password": _mask(args.pg_password),
    "s3_bucket": args.s3_bucket,
    "s3_endpoint": args.s3_endpoint or "(empty)",
    "aws_region": args.aws_region,
    "aws_access_key_id": _mask(args.aws_access_key_id),
    "aws_secret_access_key": _mask(args.aws_secret_access_key),
}
logger.info(f"use_local={args.use_local} | pg_host={args.pg_host} | s3_bucket={args.s3_bucket}")
print(json.dumps(safe, indent=2))

2026-07-16 00:00:50.121 | INFO     | __main__:<module>:71 - use_local=False | pg_host=recsys-oltp.cng4gycq4m4s.ap-southeast-1.rds.amazonaws.com | s3_bucket=recsys-ops


{
  "use_local": false,
  "train_fp": "E:\\MachineLearning\\Recommendation_System\\notebooks\\data\\001-prepare-dataset\\train.parquet",
  "val_fp": "E:\\MachineLearning\\Recommendation_System\\notebooks\\data\\001-prepare-dataset\\val.parquet",
  "meta_fp": "E:\\MachineLearning\\Recommendation_System\\notebooks\\data\\001-prepare-dataset\\raw_meta.parquet",
  "num_days_holdout": 30,
  "table_name": "movie_ratings",
  "pg_host": "recsys-oltp.cng4gycq4m4s.ap-southeast-1.rds.amazonaws.com",
  "pg_port": "5432",
  "pg_db": "postgres",
  "pg_user": "postgres",
  "pg_password": "Cu***",
  "s3_bucket": "recsys-ops",
  "s3_endpoint": "(empty)",
  "aws_region": "ap-southeast-1",
  "aws_access_key_id": "AK***",
  "aws_secret_access_key": "gu***"
}


## Load parquet inputs (from 001)

In [3]:
for fp in (args.train_fp, args.val_fp, args.meta_fp):
    if not os.path.exists(fp):
        raise FileNotFoundError(f"{fp} not found — run 001-prepare-dataset.ipynb first.")

train_df = pd.read_parquet(args.train_fp)
val_df = pd.read_parquet(args.val_fp)
metadata_df = pd.read_parquet(args.meta_fp)
logger.info(f"train={train_df.shape}  val={val_df.shape}  meta={metadata_df.shape}")
train_df.head()

2026-07-16 00:00:50.214 | INFO     | __main__:<module>:8 - train=(79425, 4)  val=(103, 4)  meta=(9742, 3)


,userId,movieId,rating,timestamp
0,429,595,5.0,828124615
1,429,225,4.0,828124615
2,429,222,4.0,828124615
3,429,218,4.0,828124615
4,429,349,3.0,828124615


## Merge item metadata + build full_df

Attach `title` and `genres` to each interaction (train + val), tag the source, then
concatenate into one frame and parse the epoch timestamp to a naive datetime.

In [4]:
meta_cols = [args.item_col, "title", "genres"]

train_features_df = pd.merge(train_df, metadata_df[meta_cols], how="left", on=args.item_col)
val_features_df = pd.merge(val_df, metadata_df[meta_cols], how="left", on=args.item_col)

full_df = (
    pd.concat(
        [
            train_features_df.assign(source="train"),
            val_features_df.assign(source="val"),
        ],
        axis=0,
        ignore_index=True,
    )
    .assign(
        # epoch seconds -> naive UTC datetime (for the OLTP TIMESTAMP column)
        timestamp=lambda d: pd.to_datetime(d[args.timestamp_col], unit="s", utc=True).dt.tz_localize(None)
    )
    .drop_duplicates(subset=[args.user_col, args.item_col, "timestamp"])
    .reset_index(drop=True)
)
logger.info(f"full_df shape={full_df.shape}  cols={list(full_df.columns)}")
full_df.head()

2026-07-16 00:00:50.267 | INFO     | __main__:<module>:22 - full_df shape=(79528, 7)  cols=['userId', 'movieId', 'rating', 'timestamp', 'title', 'genres', 'source']


,userId,movieId,rating,timestamp,title,genres,source
0,429,595,5.0,1996-03-29 18:36:55,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX,train
1,429,225,4.0,1996-03-29 18:36:55,Disclosure (1994),Drama|Thriller,train
2,429,222,4.0,1996-03-29 18:36:55,Circle of Friends (1995),Drama|Romance,train
3,429,218,4.0,1996-03-29 18:36:55,Boys on the Side (1995),Comedy|Drama,train
4,429,349,3.0,1996-03-29 18:36:55,Clear and Present Danger (1994),Action|Crime|Drama|Thriller,train


## Temporal holdout split

Rows in the last `num_days_holdout` days are NOT pushed to the OLTP — they become the
S3 holdout, to be simulated later as incoming new data (mirrors reference 02).

In [5]:
holdout_date = (full_df["timestamp"].max() - pd.to_timedelta(args.num_days_holdout, unit="D")).normalize()
logger.info(f"holdout_date = {holdout_date}")

to_insert_df = full_df.loc[lambda d: d["timestamp"] < holdout_date].copy()
holdout_df = full_df.loc[lambda d: d["timestamp"] >= holdout_date].copy()
logger.info(f"to_insert={len(to_insert_df):,}  holdout={len(holdout_df):,}")
holdout_df.head()

2026-07-16 00:00:50.284 | INFO     | __main__:<module>:2 - holdout_date = 2018-08-23 00:00:00


2026-07-16 00:00:50.290 | INFO     | __main__:<module>:6 - to_insert=79,500  holdout=28


,userId,movieId,rating,timestamp,title,genres,source
79500,305,6764,3.0,2018-08-24 15:41:46,"Rundown, The (2003)",Action|Adventure|Comedy,val
79501,305,8376,2.5,2018-08-25 07:56:30,Napoleon Dynamite (2004),Comedy,val
79502,305,122926,4.0,2018-08-25 07:57:01,Untitled Spider-Man Reboot (2017),Action|Adventure|Fantasy,val
79503,305,1215,4.0,2018-08-26 10:50:44,Army of Darkness (1993),Action|Adventure|Comedy|Fantasy|Horror,val
79504,305,56782,4.5,2018-08-28 20:29:37,There Will Be Blood (2007),Drama|Western,val


In [6]:
# Cold-start checks: every val user/item must appear in train (same contract as reference 02)
train_items = set(train_df[args.item_col].unique())
train_users = set(train_df[args.user_col].unique())
val_items = set(val_df[args.item_col].unique())
val_users = set(val_df[args.user_col].unique())

items_not_in_train = val_items - train_items
users_not_in_train = val_users - train_users
assert not items_not_in_train, f"{len(items_not_in_train)} val items not in train (cold-start)"
assert not users_not_in_train, f"{len(users_not_in_train)} val users not in train (cold-start)"
logger.info("cold-start check OK: val users/items all in train")

2026-07-16 00:00:50.333 | INFO     | __main__:<module>:11 - cold-start check OK: val users/items all in train


## Upload OLTP rows to Postgres

Local: Docker Postgres (`localhost:5435`). AWS: RDS endpoint. Falls back to Docker
defaults when creds are empty.

> `pandas.to_sql` is bypassed on purpose: pandas 2.3.3 requires SQLAlchemy>=2.0, but
> the `features` group pins apache-airflow which needs SQLAlchemy<2.0. That breaks
> pandas' SQLAlchemy detection, so it falls back to a DBAPI path that calls
> `engine.cursor()`. Fix: build DDL + INSERT by hand via `text()` (works on 1.4 & 2.0).

In [7]:
pg_user = args.pg_user or "postgres"
pg_password = args.pg_password or "postgres"
# AWS RDS requires SSL (rds.force_ssl=1 in ap-southeast-1). Local Docker does not.
sslmode = "require" if not args.use_local else "disable"
conn_str = f"postgresql+psycopg2://{pg_user}:{pg_password}@{args.pg_host}:{args.pg_port}/{args.pg_db}?sslmode={sslmode}"
logger.info(f"connecting to {args.pg_host}:{args.pg_port}/{args.pg_db} (local={args.use_local}, sslmode={sslmode})")
engine = create_engine(conn_str)

PANDAS_TO_SQL = {
    "int64": "BIGINT", "int32": "INTEGER", "Int64": "BIGINT",
    "float64": "DOUBLE PRECISION", "float32": "REAL",
    "bool": "BOOLEAN", "object": "TEXT", "string": "TEXT",
    "datetime64[ns]": "TIMESTAMP", "category": "TEXT",
}


def ddl_for(df, schema, table):
    cols = [f'    "{c}" {PANDAS_TO_SQL.get(str(df[c].dtype), "TEXT")}' for c in df.columns]
    return f"CREATE TABLE {schema}.{table} (\n" + ",\n".join(cols) + "\n)"


fq = f"{args.pg_schema}.{args.table_name}"
cols = list(to_insert_df.columns)
col_names = [f'"{c}"' for c in cols]

# Bulk insert via psycopg2.extras.execute_values: ONE round trip (chunked) instead of
# one round trip per row (executemany). 79k rows over a WAN link to RDS would otherwise
# time out. execute_values needs a real DBAPI cursor (mogrify lives on the cursor, not
# the connection), so we pull the raw psycopg2 conn out of the SA transaction and open a
# cursor on it. The cursor shares the SA-managed transaction.
from psycopg2.extras import execute_values

with engine.begin() as conn:
    conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {args.pg_schema};"))
    conn.execute(text(f"DROP TABLE IF EXISTS {fq};"))
    conn.execute(text(ddl_for(to_insert_df, args.pg_schema, args.table_name)))
    if len(to_insert_df):
        dbapi = conn.connection  # raw psycopg2 connection inside the SA transaction
        insert_sql = f"INSERT INTO {fq} ({', '.join(col_names)}) VALUES %s"
        tuples = [tuple(row) for row in to_insert_df.itertuples(index=False, name=None)]
        with dbapi.cursor() as cur:
            execute_values(cur, insert_sql, tuples, page_size=1000)
logger.info(f"wrote {len(to_insert_df):,} rows to {fq} (execute_values bulk)")

2026-07-16 00:00:50.366 | INFO     | __main__:<module>:6 - connecting to recsys-oltp.cng4gycq4m4s.ap-southeast-1.rds.amazonaws.com:5432/postgres (local=False, sslmode=require)


2026-07-16 00:00:57.992 | INFO     | __main__:<module>:43 - wrote 79,500 rows to public.movie_ratings (execute_values bulk)


In [8]:
# Add a primary key on the detected key columns.
# PK = (userId, movieId, timestamp) — unique per rating event.
pk_cols = [args.user_col, args.item_col, "timestamp"]
with engine.begin() as conn:
    conn.execute(text(
        f'ALTER TABLE {args.pg_schema}.{args.table_name} '
        f"ADD CONSTRAINT {args.table_name}_pkey PRIMARY KEY ({', '.join(chr(34) + c + chr(34) for c in pk_cols)});"
    ))
logger.info(f"PK on {pk_cols}")

2026-07-16 00:00:58.227 | INFO     | __main__:<module>:9 - PK on ['userId', 'movieId', 'timestamp']


## Create mirror table for new-data simulation

Like the reference: a `new_<table>` with the same schema, used later to simulate incoming new data.

In [9]:
new_table = f"new_{args.table_name}"
with engine.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS {args.pg_schema}.{new_table};"))
    conn.execute(text(
        f"CREATE TABLE {args.pg_schema}.{new_table} "
        f"(LIKE {args.pg_schema}.{args.table_name} INCLUDING ALL);"
    ))
logger.info(f"created mirror table {args.pg_schema}.{new_table}")

2026-07-16 00:00:58.438 | INFO     | __main__:<module>:8 - created mirror table public.new_movie_ratings


## Upload holdout to S3 / MinIO

In [10]:
holdout_df.sort_values("timestamp", ascending=True).to_parquet(args.holdout_fp, index=False)
logger.info(f"wrote local {args.holdout_fp} ({len(holdout_df):,} rows)")

if args.use_local:
    s3 = boto3.client(
        "s3",
        aws_access_key_id=args.aws_access_key_id or "admin",
        aws_secret_access_key=args.aws_secret_access_key or "Password1234",
        endpoint_url=args.s3_endpoint or "http://localhost:9000",
    )
    try:
        s3.create_bucket(Bucket=args.s3_bucket)
    except Exception as exc:  # bucket may already exist
        logger.info(f"create_bucket skipped: {exc}")
else:
    s3 = boto3.client(
        "s3",
        aws_access_key_id=args.aws_access_key_id,
        aws_secret_access_key=args.aws_secret_access_key,
        region_name=args.aws_region,
    )

s3_key = os.path.basename(args.holdout_fp)
s3.upload_file(args.holdout_fp, args.s3_bucket, s3_key)
logger.info(f"uploaded to s3://{args.s3_bucket}/{s3_key} (local={args.use_local})")

2026-07-16 00:00:58.460 | INFO     | __main__:<module>:2 - wrote local holdout.parquet (28 rows)


2026-07-16 00:00:59.607 | INFO     | __main__:<module>:25 - uploaded to s3://recsys-ops/holdout.parquet (local=False)


## Verify

In [11]:
with engine.connect() as conn:
    n_rows = conn.execute(text(f"SELECT COUNT(*) FROM {args.pg_schema}.{args.table_name}")).scalar()
    n_tables = conn.execute(text(
        f"SELECT COUNT(*) FROM information_schema.tables "
        f"WHERE table_schema='{args.pg_schema}' AND table_name LIKE '{args.table_name}%';"
    )).scalar()
logger.info(f"Postgres {args.pg_schema}.{args.table_name}: {n_rows:,} rows | matching tables: {n_tables}")

objs = s3.list_objects_v2(Bucket=args.s3_bucket)
keys = [o["Key"] for o in objs.get("Contents", [])]
logger.info(f"S3 bucket {args.s3_bucket} objects: {keys}")

assert n_rows == len(to_insert_df), "PG row count mismatch!"
assert s3_key in keys, f"{s3_key} not found in S3!"
logger.info("verification OK")

2026-07-16 00:00:59.831 | INFO     | __main__:<module>:7 - Postgres public.movie_ratings: 79,500 rows | matching tables: 1


2026-07-16 00:00:59.914 | INFO     | __main__:<module>:11 - S3 bucket recsys-ops objects: ['holdout.parquet']


2026-07-16 00:00:59.916 | INFO     | __main__:<module>:15 - verification OK


## Summary

- Loaded `train.parquet` + `val.parquet` + `raw_meta.parquet` from 001.
- Merged movie metadata (title, genres) into interactions.
- Temporal holdout: last 30 days → S3, the rest → Postgres OLTP.
- Cold-start checks (val users/items ⊆ train).
- Uploaded OLTP rows → Postgres `public.movie_ratings` (+ mirror `new_movie_ratings`).
- Uploaded holdout parquet → S3/MinIO `recsys-ops/holdout.parquet`.
- Verified row count + object presence.

Switch local ↔ AWS by flipping `DATA_UPLOAD_USE_LOCAL` in `.env` (see `docs/README.md`).